In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [ ]:
base_path = "/mnt/AEGIS/AEGIS/Deidentification"

target_dirs = ['AEGIS-Order-Med1',
 'AEGIS-Basic-Order',
 'AEGIS-Weight-and-Height',
 'AEGIS-Nursing-Initial-Assessment',
 'AEGIS-Dx-KCD',
 'AEGIS-Order-CIV2',
 'AEGIS-Order-Test',
 'AEGIS-Order-CIV1',
 'AEGIS-Administer-Med',
 'AEGIS-Flowsheet-Value',
 'AEGIS-Basic-Info',
 'AEGIS-Dx-SNOMED']

# pt_id 수집
all_pt_ids = set()

for folder in target_dirs:
    target_path = os.path.join(base_path, folder, 'de')
    csv_files = glob.glob(os.path.join(target_path, '**/*.csv'),recursive = True)
    
    print(f"[{folder}]에서 pt_id 수집 중...")
    
    for file in csv_files:
        try:
            df = pd.read_csv(file, usecols = ['pt_id'], dtype = {"pt_id":str} ,sep=",", quotechar ='"', lineterminator = '\n')
            pt_ids = df['pt_id'].dropna().unique()
            all_pt_ids.update(pt_ids)
            
        except ValueError as ve:
            print(f"컬럼 없음: {file}")
        except Exception as e:
            print(f"[오류] {file} -> {e}")
            raise
            
            
print(len(all_pt_ids))


In [ ]:
# pt_id 노이즈 테이블            
pt_ids = sorted(all_pt_ids)
np.random.seed(1)

offset_days = np.random.choice([-7,7], size = len(pt_ids)) # -7 or +7일
offset_minutes = np.random.randint(-30, 31, size = len(pt_ids)) # -30분 ~ +30분

mapping_df = pd.DataFrame({
    'pt_id': pt_ids,
    'new_pt_id': list(range(1, len(pt_ids)+1)),
    'date_noise': offset_days,
    'time_noise': offset_minutes
})

mapping_df

In [ ]:
mapping_df.to_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", index = False, encoding = 'utf-8')
mapping_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n')
